In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("naserabdullahalam/phishing-email-dataset")

print("Path to dataset files:", path)

100%|██████████| 77.1M/77.1M [00:00<00:00, 210MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/naserabdullahalam/phishing-email-dataset/versions/1


In [2]:
import os
import pandas as pd

# List the files in the downloaded dataset directory
files = os.listdir(path)
print("Files in dataset directory:", files)

# Identify CSV files
csv_files = [f for f in files if f.endswith('.csv')]
print("CSV files found:", csv_files)

# Merge all CSV files if multiple exist, or copy/save the main one
if len(csv_files) == 1:
    df = pd.read_csv(os.path.join(path, csv_files[0]))
    df.to_csv('phishing_dataset.csv', index=False)
    print("Saved single dataset to 'phishing_dataset.csv'")
elif len(csv_files) > 1:
    dfs = []
    for f in csv_files:
        file_path = os.path.join(path, f)
        try:
            temp_df = pd.read_csv(file_path)
            dfs.append(temp_df)
        except Exception as e:
            print(f"Error reading {f}: {e}")

    # Combine all dataframes
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_df.to_csv('phishing_dataset.csv', index=False)
    print(f"Merged {len(dfs)} CSV files into 'phishing_dataset.csv'")
else:
    print("No CSV files found in the directory. Checking subdirectories...")


Files in dataset directory: ['Nigerian_Fraud.csv', 'Enron.csv', 'phishing_email.csv', 'Nazario.csv', 'SpamAssasin.csv', 'Ling.csv', 'CEAS_08.csv']
CSV files found: ['Nigerian_Fraud.csv', 'Enron.csv', 'phishing_email.csv', 'Nazario.csv', 'SpamAssasin.csv', 'Ling.csv', 'CEAS_08.csv']
Merged 7 CSV files into 'phishing_dataset.csv'


In [3]:
# Preview the merged dataset
if os.path.exists('phishing_dataset.csv'):
    merged_df = pd.read_csv('phishing_dataset.csv')
    print(f"Dataset shape: {merged_df.shape}")
    display(merged_df.head())


Dataset shape: (164972, 8)


/tmp/ipykernel_1780/1030687332.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv('phishing_dataset.csv')


,sender,receiver,date,subject,body,urls,label,text_combined
0,MR. JAMES NGOLA. <james_ngola2002@maktoob.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 02:38:20 +0000",URGENT BUSINESS ASSISTANCE AND PARTNERSHIP,FROM:MR. JAMES NGOLA.\nCONFIDENTIAL TEL: 233-2...,0.0,1,NaN
1,Mr. Ben Suleman <bensul2004nng@spinfinder.com>,R@M,"Thu, 31 Oct 2002 05:10:00 -0000",URGENT ASSISTANCE /RELATIONSHIP (P),"Dear Friend,\n\nI am Mr. Ben Suleman a custom ...",0.0,1,NaN
2,PRINCE OBONG ELEME <obong_715@epatra.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 22:17:55 +0100",GOOD DAY TO YOU,FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,0.0,1,NaN
3,PRINCE OBONG ELEME <obong_715@epatra.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 22:44:20 -0000",GOOD DAY TO YOU,FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,0.0,1,NaN
4,Maryam Abacha <m_abacha03@www.com>,R@M,"Fri, 01 Nov 2002 01:45:04 +0100",I Need Your Assistance.,"Dear sir, \n \nIt is with a heart full of hope...",0.0,1,NaN


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset and handle potential mixed/null values
df = pd.read_csv('phishing_dataset.csv', low_memory=False)

# Let's combine available text columns to get the best representation of each email
# Depending on the source CSV, the text might be in 'body', 'subject', or 'text_combined'
df['email_text'] = df['body'].fillna('') + " " + df['subject'].fillna('') + " " + df['text_combined'].fillna('')

# Ensure labels are binary integers and drop rows without valid labels
df = df.dropna(subset=['label'])
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# Filter out empty text rows
df = df[df['email_text'].str.strip() != '']

# Split the dataset (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    df['email_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")


Training set size: 131976
Testing set size: 32995


In [5]:
# 1. Rules-based baseline model
def rules_baseline_predict(text):
    # Simple check for common phishing keywords
    phishing_keywords = ['urgent', 'winner', 'money', 'free', 'verify', 'account', 'bank', 'click', 'suspend', 'security', 'transfer']
    text_lower = str(text).lower()
    for word in phishing_keywords:
        if word in text_lower:
            return 1 # Phishing
    return 0 # Safe

# Evaluate the Rules-based baseline model
baseline_preds = [rules_baseline_predict(text) for text in X_test]
baseline_acc = accuracy_score(y_test, baseline_preds)

print("=== Rules-Based Baseline Model ===")
print(f"Accuracy: {baseline_acc:.4f}")
print(classification_report(y_test, baseline_preds))


=== Rules-Based Baseline Model ===
Accuracy: 0.5747
              precision    recall  f1-score   support

           0       0.54      0.72      0.62     15838
           1       0.63      0.44      0.52     17157

    accuracy                           0.57     32995
   macro avg       0.59      0.58      0.57     32995
weighted avg       0.59      0.57      0.57     32995



In [7]:
# Vectorize the text using TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 2. TF-IDF + Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

print("=== TF-IDF + Logistic Regression ===")
print(f"Accuracy: {lr_acc:.4f}")
print(classification_report(y_test, lr_preds))

=== TF-IDF + Logistic Regression ===
Accuracy: 0.9871
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     15838
           1       0.99      0.99      0.99     17157

    accuracy                           0.99     32995
   macro avg       0.99      0.99      0.99     32995
weighted avg       0.99      0.99      0.99     32995



In [8]:
# 3. TF-IDF + Linear SVM
svm_model = LinearSVC(random_state=42, max_iter=2000)
svm_model.fit(X_train_tfidf, y_train)

svm_preds = svm_model.predict(X_test_tfidf)
svm_acc = accuracy_score(y_test, svm_preds)

print("=== TF-IDF + Linear SVM ===")
print(f"Accuracy: {svm_acc:.4f}")
print(classification_report(y_test, svm_preds))

=== TF-IDF + Linear SVM ===
Accuracy: 0.9942
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     15838
           1       0.99      1.00      0.99     17157

    accuracy                           0.99     32995
   macro avg       0.99      0.99      0.99     32995
weighted avg       0.99      0.99      0.99     32995



In [9]:
import joblib

# Save the models and vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')
joblib.dump(lr_model, 'logistic_regression_model.joblib')
joblib.dump(svm_model, 'linear_svm_model.joblib')

print("Successfully saved:")
print("- tfidf_vectorizer.joblib")
print("- logistic_regression_model.joblib")
print("- linear_svm_model.joblib")

Successfully saved:
- tfidf_vectorizer.joblib
- logistic_regression_model.joblib
- linear_svm_model.joblib


In [10]:
app_code = """
import os
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Load the vectorizer and models
vectorizer = joblib.load('tfidf_vectorizer.joblib')
lr_model = joblib.load('logistic_regression_model.joblib')
svm_model = joblib.load('linear_svm_model.joblib')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    emails = data.get('emails', [])
    model_type = data.get('model', 'svm')  # 'lr' or 'svm'

    if not emails:
        return jsonify({'error': 'No email texts provided'}), 400

    # Transform texts
    features = vectorizer.transform(emails)

    # Select model
    if model_type == 'lr':
        predictions = lr_model.predict(features)
        probabilities = lr_model.predict_proba(features)[:, 1].tolist()
    else:
        predictions = svm_model.predict(features)
        # LinearSVC doesn't support predict_proba by default; use decision_function instead
        probabilities = svm_model.decision_function(features).tolist()

    results = []
    for idx, (pred, score) in enumerate(zip(predictions, probabilities)):
        results.append({
            'email_index': idx,
            'text_preview': emails[idx][:100],
            'prediction': int(pred),
            'label_name': 'Phishing' if pred == 1 else 'Safe',
            'confidence_score': score
        })

    return jsonify({
        'model_used': 'Linear SVM' if model_type == 'svm' else 'Logistic Regression',
        'results': results
    })

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("Flask application file 'app.py' has been successfully created.")


Flask application file 'app.py' has been successfully created.


In [11]:
# Example of how to run this server in the background and test it
import subprocess
import time
import requests
import json
import pandas as pd

# Start Flask app in the background
print("Starting Flask API...")
proc = subprocess.Popen(['python', 'app.py'])
time.sleep(3) # Wait for the server to spin up

# Sample user payload
payload = {
    'model': 'svm',
    'emails': [
        "Dear customer, please click this link to verify your bank account details immediately.",
        "Hi team, are we still on for the project alignment meeting scheduled at 3:00 PM today?"
    ]
}

# Query the API
try:
    response = requests.post('http://127.0.0.1:5000/predict', json=payload)
    response_json = response.json()
    print("\n--- API Response ---")
    print(json.dumps(response_json, indent=2))

    # Exporting output to a CSV file
    results_df = pd.DataFrame(response_json['results'])
    results_df['model_used'] = response_json['model_used']
    results_df.to_csv('api_predictions_export.csv', index=False)
    print("\nPredictions exported successfully to 'api_predictions_export.csv'")

except Exception as e:
    print("Failed to connect to the Flask server:", e)

finally:
    # Terminate the background API server process
    proc.terminate()


Starting Flask API...

--- API Response ---
{
  "model_used": "Linear SVM",
  "results": [
    {
      "confidence_score": 2.9047403772036224,
      "email_index": 0,
      "label_name": "Phishing",
      "prediction": 1,
      "text_preview": "Dear customer, please click this link to verify your bank account details immediately."
    },
    {
      "confidence_score": -2.3020197620073586,
      "email_index": 1,
      "label_name": "Safe",
      "prediction": 0,
      "text_preview": "Hi team, are we still on for the project alignment meeting scheduled at 3:00 PM today?"
    }
  ]
}

Predictions exported successfully to 'api_predictions_export.csv'


### Inspecting the generated Flask application (`app.py`)
Here is the exact code stored in `app.py`:

In [12]:
with open('app.py', 'r') as f:
    print(f.read())


import os
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Load the vectorizer and models
vectorizer = joblib.load('tfidf_vectorizer.joblib')
lr_model = joblib.load('logistic_regression_model.joblib')
svm_model = joblib.load('linear_svm_model.joblib')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    emails = data.get('emails', [])
    model_type = data.get('model', 'svm')  # 'lr' or 'svm'
    
    if not emails:
        return jsonify({'error': 'No email texts provided'}), 400
        
    # Transform texts
    features = vectorizer.transform(emails)
    
    # Select model
    if model_type == 'lr':
        predictions = lr_model.predict(features)
        probabilities = lr_model.predict_proba(features)[:, 1].tolist()
    else:
        predictions = svm_model.predict(features)
        # LinearSVC doesn't support predict_proba by default; use decision_function instead
        proba

### Inspecting Saved Models and Vectorizer Metadata
Below we load and inspect key parameters of our joblib files directly in the code output:

In [13]:
import joblib

# Load components
loaded_vectorizer = joblib.load('tfidf_vectorizer.joblib')
loaded_lr = joblib.load('logistic_regression_model.joblib')
loaded_svm = joblib.load('linear_svm_model.joblib')

print("=== TF-IDF Vectorizer Properties ===")
print(f"Number of features: {len(loaded_vectorizer.vocabulary_)}")
print(f"Stop words: {loaded_vectorizer.stop_words}")
print(f"Sample features (first 10 items): {list(loaded_vectorizer.vocabulary_.keys())[:10]}\n")

print("=== Logistic Regression Model Properties ===")
print(f"Classes: {loaded_lr.classes_}")
print(f"Coefficients shape: {loaded_lr.coef_.shape}")
print(f"Intercept: {loaded_lr.intercept_}\n")

print("=== Linear SVM Model Properties ===")
print(f"Classes: {loaded_svm.classes_}")
print(f"Coefficients shape: {loaded_svm.coef_.shape}")
print(f"Intercept: {loaded_svm.intercept_}")

=== TF-IDF Vectorizer Properties ===
Number of features: 10000
Stop words: english
Sample features (first 10 items): ['dear', 'summer', 'best', 'time', 'break', 'work', 'think', 'health', 'personal', 'life']

=== Logistic Regression Model Properties ===
Classes: [0 1]
Coefficients shape: (1, 10000)
Intercept: [0.70154163]

=== Linear SVM Model Properties ===
Classes: [0 1]
Coefficients shape: (1, 10000)
Intercept: [0.42992786]
